# Tutorial - Three-Phase Induction Motor (TPIM) - Part 7
======================

## Section 3: Variable Frequency Drive (V/f Inverter)

This notebook continues **tutorial_part_6.ipynb** (Sections 1-2, driven by an ideal `SourceAC`) and covers Section 3: driving the `MotorElement` with a three-phase voltage source inverter under scalar V/f control (`InverterVF`), through `.run_with_inverter_vf()`.

This notebook is self-contained and can be run independently.

In [ ]:
import ross as rs
import numpy as np

from ross.units import Q_

# Make sure the default renderer is set to 'notebook' for inline plots in Jupyter
import plotly.io as pio
import plotly.graph_objects as go

pio.renderers.default = "notebook"

## 3.1 Instantiating the Motor

The motor parameters are the same as in Section 2 of tutorial_part_6.ipynb (1.5 hp / 127 V / 60 Hz / 4 poles / 1710 RPM). Note that, when the motor is driven through `.run_with_inverter_vf()`, `voltage_net` and `frequency_net` do not need to be informed at instantiation: the `InverterVF` internally sets the voltage and frequency applied to the motor according to the V/f reference.

In [ ]:
motor3 = rs.MotorElement(
    n=0,
    tag="TPIM_VF",
    power_nom=Q_(1.5, "hp"),
    voltage_nom=127,
    speed_nom=Q_(1710, "RPM"),
    frequency_nom=Q_(60.0, "Hz"),
    n_poles=4,
    stator_resistance=2.5,
    rotor_resistance=1.8,
    stator_reactance=1.3,
    rotor_reactance=1.3,
    mutual_reactance=43.08,
    Ip_motor=0.0372,
    viscosity_coeff=0.0,
    Ip_load=0.0,
)
motor3

## 3.2 Running with `run_with_inverter_vf`

The `.run_with_inverter_vf()` method simulates the motor driven by a three-phase voltage source inverter (`InverterVF`) using Space Vector PWM (SVPWM) modulation with scalar V/f speed control. The applied voltage is reduced proportionally to the reference frequency, keeping the air-gap flux approximately constant. Besides the arguments already seen in `.run_with_AC_source()`, the relevant inverter parameters are:

- `frequency_s`: IGBT switching frequency (`Fs`);
- `time_ramp`: acceleration ramp time, during which the reference frequency increases linearly from zero to `frequency_ref` [s];
- `frequency_ref`: V/f reference frequency. If `None`, half of the motor nominal frequency is used.

In this example, the inverter is set to a reference frequency of **30 Hz** (half of the nominal frequency), so the steady-state speed is expected to be approximately half of the nominal speed. The switching frequency `Fs` is set to 5000 Hz and is reused below to scale the FFT plots' frequency range.

**Note on `time_step`:** the internal SVPWM carrier is reconstructed from samples taken every `time_step`. This must be small compared to the switching period `Ts = 2*pi/frequency_s` (here `Ts` = 200 microseconds) - as a rule of thumb, at least a few dozen samples per switching period - otherwise the carrier is aliased and the synthesized voltage can come out systematically wrong. `time_step=1e-5` below gives 20 samples per switching period.

In [ ]:
Fs = 5000.0  # IGBT switching frequency [Hz], reused below for the FFT plots

dt = 1e-3
tf = 3.0
t3 = np.arange(0, tf + dt, dt)

results3 = motor3.run_with_inverter_vf(
    t3,
    time_step=1e-5,
    load_torque_entrance_time=1.5,
    load_torque_ratio=1.0,
    frequency_s=Q_(Fs, "Hz"),
    time_ramp=0.6667,
    frequency_ref=Q_(30.0, "Hz"),
)

## 3.3 Time-Domain Results

### Electromagnetic Torque

In [ ]:
results3.plot_torque().show()

### Rotor Speed

The acceleration ramp followed by the steady-state speed, around half of the nominal speed (1710 RPM), can be observed below.

In [ ]:
results3.plot_speed().show()

### Stator Phase Currents

In [ ]:
results3.plot_phase_currents(reference_frame="a-b-c").show()

### Stator Phase Voltages

The voltage waveforms show the characteristic SVPWM pattern generated by `InverterVF`.

In [ ]:
results3.plot_phase_voltages().show()

### Stator Line Voltages

In [ ]:
results3.plot_line_voltages().show()

## 3.4 Frequency-Domain Results (FFT)

For signals coming from the inverter, the spectrum reveals the harmonic components generated by the SVPWM modulation, concentrated around multiples of the switching frequency `Fs`. To keep these plots focused on the relevant content, every FFT figure below restricts the displayed band to **0.5 Hz - 2.1 x Fs** (via the `frequency_range` argument): the lower bound of 0.5 Hz discards the (irrelevant) DC bin, while the upper bound of 2.1 x Fs comfortably includes the fundamental, the switching frequency itself and its first sidebands, without cluttering the plot with content far above the switching frequency.

### FFT of the Electromagnetic Torque

In [ ]:
results3.plot_torque(
    domain="frequency",
    frequency_range=Q_((0.5, 2.1 * Fs), "Hz"),
).show()

### FFT of the Stator Phase Currents

In [ ]:
results3.plot_phase_currents(
    domain="frequency",
    frequency_range=Q_((0.5, 2.1 * Fs), "Hz"),
).show()

### FFT of the Stator Line Voltages

In [ ]:
results3.plot_line_voltages(
    domain="frequency",
    frequency_range=Q_((0.5, 2.1 * Fs), "Hz"),
).show()

## 3.5 Reference Frames - Park (d-q)

The stator currents can also be displayed in the Park (d-q) reference frame, which is useful for analyzing the flux and torque-producing current components.

In [ ]:
results3.plot_phase_currents(reference_frame="d-q").show()

In [ ]:
results3.plot_phase_currents(
    reference_frame="d-q",
    domain="frequency",
    frequency_range=Q_((0.5, 2.1 * Fs), "Hz"),
).show()

---
Continue to **tutorial_part_8.ipynb** for the Field-Oriented Control (`InverterFOC`, iFOC) example, and a comparison between V/f and FOC speed control.